# Adaptive pipeline on a GPU (Colab or Kaggle)

Runs `python -m ondemand.adaptive run` with the models hosted inside the run: ColVec in-process, KDL and Chandra
through vLLM servers that the run starts and stops itself, one model at a time. Every model writes a per-page cache,
`timings.jsonl` and `logs/<service>.log` (on Colab those folders live on Drive); the run writes `per_query.jsonl` line by line, so a disconnect loses nothing
that finished. Rerun the run cell to continue (`--resume`).

Input: `adaptive_bundle.zip` from `python -m ondemand bundle --target adaptive` (code, corpus, light-prep pages, gate,
embedding caches, the corpus ColVec matrix and KDL pages for the reproduction checks).

GPU: an L4 (24 GB) is the reference. KDL needs bf16 and about 20 GB; ColVec-8b about 17 GB. On a T4 use
`--colvec-8bit` or the 4b model and point KDL/Chandra at a remote server, or leave them cached.

In [ ]:
!nvidia-smi

## 1. Settings

In [ ]:
TAG          = "gpu_rule_v1"
ROUTER       = "rule"
LIMIT        = 0            # >0: only the first N queries (a smoke run)
SERVICES     = ["--colvec", "local", "--kdl", "vllm", "--chandra", "vllm"]   # ppocr stays cached: light prep is a separate one-time step
QA           = False        # generator + judge through OpenRouter (needs OPENROUTER_API_KEY)
ALLOW_API    = False        # te3-small embeddings for chunk texts not in the cache (new KDL/Chandra text needs this for --ranker hybrid)
RANKER       = "hybrid"
VERIFY_QUERIES = 20         # ColVec reproduction gate against the corpus matrix
INSTALL_PADDLE = False      # only for --ppocr local or `prep`
BUNDLE_NAME  = "adaptive_bundle.zip"
DRIVE_FOLDER = "FOR-COLAB/Colab-for-Axiom-DE-RnD"

## 2. Code, corpus and a persistent work folder

In [ ]:
import glob, hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

KAGGLE = Path("/kaggle/input").exists()
REPO = Path("/kaggle/working/AXIOM_DE-RD" if KAGGLE else "/content/AXIOM_DE-RD")
REPO.mkdir(parents=True, exist_ok=True)
DRIVE = None
if KAGGLE:
    found = glob.glob(f"/kaggle/input/**/{BUNDLE_NAME}", recursive=True)
    if not found:
        raise SystemExit(f"attach a Kaggle dataset holding {BUNDLE_NAME}")
    bundle_zip = Path(found[0])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    bundle_zip = DRIVE / BUNDLE_NAME
    if not bundle_zip.exists():
        from google.colab import files
        name = next(n for n in files.upload() if n.endswith(".zip"))
        bundle_zip = Path("/content") / BUNDLE_NAME
        shutil.move(name, bundle_zip)
        shutil.copy(bundle_zip, DRIVE / BUNDLE_NAME)

if not (REPO / "BUNDLE_MANIFEST.json").exists():
    shutil.copy(bundle_zip, "/tmp/bundle.zip")
    with zipfile.ZipFile("/tmp/bundle.zip") as zf:
        zf.extractall(REPO)
manifest = json.loads((REPO / "BUNDLE_MANIFEST.json").read_text())
assert manifest["target"] == "adaptive", manifest["target"]
FP = manifest["bundle_fingerprint"]
WORK = REPO / "data/work/ondemand_v2" / FP
print(f"bundle {FP}: {manifest['n_documents']} documents, {manifest['n_pages']} pages, git {manifest['git']['sha'][:8]}")

STARTED = __import__("time").time()
WRITTEN = ["results", "logs", "colvec/live", "kdl/live", "chandra/live", "ppocr/live"]
if DRIVE:
    persistent = DRIVE / FP / "adaptive_work"
    for rel in WRITTEN:
        local, remote = WORK / rel, persistent / rel
        remote.mkdir(parents=True, exist_ok=True)
        if local.is_symlink():
            continue
        if local.exists():
            shutil.copytree(local, remote, dirs_exist_ok=True)
            shutil.rmtree(local)
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(remote, target_is_directory=True)
    print("results, logs and model caches on Drive:", persistent)
else:
    print("work folder:", WORK, "(Kaggle keeps /kaggle/working when you save a version)")
os.chdir(REPO)

## 3. Install

The main env gets the ColVec pins (from `colab/01`) and the clients. vLLM 0.19.0 pins its own torch/transformers, so it
lives in its own venv and the run calls its `vllm` binary (`VLLM_BIN`). PaddleOCR clashes with torch's CUDA libraries,
so it gets a venv too (`PPOCR_PYTHON`).

In [ ]:
import signal

def sh(cmd, env=None):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ, **(env or {})})
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        print("interrupted: asking the run to stop its models and servers...", flush=True)
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(timeout=300)
        except subprocess.TimeoutExpired:
            proc.kill()
        raise
    if proc.wait():
        raise RuntimeError(f"exit code {proc.returncode}")

sh(f"{sys.executable} -m pip install -q 'transformers==5.14.1' 'sentence-transformers==5.6.0' 'accelerate==1.14.0' "
   "pymupdf PyStemmer 'pdf-inspector>=0.2.6,<0.3' 'chandra-ocr>=0.2,<0.3' 'pytrec_eval-terrier>=0.5.10' "
   "'Markdown>=3.0' scipy pydantic python-dotenv httpx requests tiktoken uv")
if not Path("/opt/vllm-env/bin/vllm").exists():
    sh("uv venv /opt/vllm-env --python python3 && uv pip install --python /opt/vllm-env/bin/python 'vllm==0.19.0'")
if INSTALL_PADDLE and not Path("/opt/paddle-env/bin/python").exists():
    sh("uv venv /opt/paddle-env --python python3 && uv pip install --python /opt/paddle-env/bin/python "
       "paddlepaddle-gpu==3.0.0 --index-url https://www.paddlepaddle.org.cn/packages/stable/cu126/ "
       "--extra-index-url https://pypi.org/simple && uv pip install --python /opt/paddle-env/bin/python "
       "'paddleocr>=3.0,<4' pymupdf")

from importlib.metadata import version
pins = {"transformers": "5.14.1", "accelerate": "1.14.0", "sentence-transformers": "5.6.0"}
wrong = {k: version(k) for k, v in pins.items() if version(k) != v}
if wrong:
    raise SystemExit(f"installed versions differ from the pins: {wrong}. Restart the session and rerun from section 2.")

## 4. Environment and doctor

In [ ]:
os.environ["PYTHONPATH"] = str(REPO)
os.environ["VLLM_BIN"] = "/opt/vllm-env/bin/vllm"
if Path("/opt/paddle-env/bin/python").exists():
    os.environ["PPOCR_PYTHON"] = "/opt/paddle-env/bin/python"
try:
    if KAGGLE:
        from kaggle_secrets import UserSecretsClient
        os.environ["OPENROUTER_API_KEY"] = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
    else:
        from google.colab import userdata
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception as error:
    print("no OPENROUTER_API_KEY secret:", type(error).__name__, "(needed only for QA or uncached embeddings)")

ADAPTIVE = [sys.executable, "-m", "ondemand.adaptive"]
sh(ADAPTIVE + ["doctor"])

## 5. Reproduction checks

ColVec: rescore the gate pages of the first `VERIFY_QUERIES` queries in-process and compare with the corpus matrix
(the bar from `colab/01`: Pearson >= 0.999 and top-1 agreement >= 90%). KDL: parse a few pages through the self-hosted
vLLM server and compare with the corpus parse. Both write to the same caches the run reads, so this work is not wasted.

In [ ]:
sh(ADAPTIVE + ["call", "colvec", "--colvec", "local", "--verify", str(VERIFY_QUERIES), "--out", "/tmp/colvec_verify.json"])
check = json.loads(Path("/tmp/colvec_verify.json").read_text())
print(json.dumps(check, indent=1))
assert check.get("reproduces"), "ColVec does not reproduce the corpus matrix: check the pins before the full run"

In [ ]:
import difflib
corpus = {json.loads(l)["page_id"]: json.loads(l)["text"] for l in open(WORK / "kdl/kdl_pages.jsonl")}
probe = [p for p in sorted(corpus) if corpus[p].strip()][::997][:5]
sh(ADAPTIVE + ["call", "kdl", "--kdl", "vllm", "--pages", *probe, "--out", "/tmp/kdl_probe.json"])
live = json.loads(Path("/tmp/kdl_probe.json").read_text())["text"]
for p in probe:
    print(f"{difflib.SequenceMatcher(None, corpus[p], live.get(p, '')).ratio():.3f}  {p}")

In [ ]:
if "chandra" in " ".join(SERVICES):
    page = probe[0]
    sh(ADAPTIVE + ["call", "chandra", *SERVICES, "--pages", page, "--out", "/tmp/chandra_probe.json"])
    text = json.loads(Path("/tmp/chandra_probe.json").read_text())["text"].get(page, "")
    print(len(text), "characters")
    print(text[:1500])

## 6. Smoke run: 3 queries, each model handed the GPU in turn

The checks above run each model in its own process. A run loads ColVec, releases it and starts vLLM in one process;
this proves that handoff on this GPU before the full run.

In [ ]:
cmd = ADAPTIVE + ["run", "--tag", TAG + "_smoke", "--router", "fixed:visual", "--ranker", RANKER, "--limit", "3",
                  "--resume", *SERVICES]
sh(cmd + (["--allow-api"] if ALLOW_API else []))

## 7. Run (rerun this cell after a disconnect: it resumes)

In [ ]:
cmd = ADAPTIVE + ["run", "--tag", TAG, "--router", ROUTER, "--ranker", RANKER, "--exclude-flawed", "--resume", *SERVICES]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if QA:
    cmd += ["--qa"]
if ALLOW_API or QA:
    cmd += ["--allow-api"]
sh(cmd)

## 8. Summary and export

In [ ]:
import collections
out = WORK / "results" / TAG
metrics = json.loads((out / "metrics.json").read_text())
print(json.dumps({k: metrics[k] for k in ("n_queries", "branch_counts", "fallbacks", "cost", "metrics")}, indent=1))

def timing_summary(path):
    rows = [json.loads(l) for l in open(path) if l.strip()]
    by = collections.defaultdict(list)
    for r in rows:
        by[r["stage"]].append(r["seconds"] / max(r["n"], 1))
    return {s: {"records": len(v), "per_unit_mean": round(sum(v) / len(v), 4)} for s, v in by.items()}

for path in sorted(WORK.glob("*/live/*/timings.jsonl")):
    print(path.relative_to(WORK), json.dumps(timing_summary(path)))

keep = ["results/" + TAG, "logs"] + [str(p.parent.relative_to(WORK)) for p in WORK.glob("*/live/*/settings.json")]
with zipfile.ZipFile(f"/tmp/{TAG}_export.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for rel in keep:
        for f in (WORK / rel).rglob("*"):
            if f.is_file() and "subpdf" not in f.parts:
                zf.write(f, f"data/work/ondemand_v2/{FP}/{f.relative_to(WORK)}")
    new_api = [f for rel in ("embedding_cache", "qa_cache") for f in (WORK / rel).rglob("*")
               if f.is_file() and f.stat().st_mtime > STARTED]
    for f in new_api:
        zf.write(f, f"data/work/ondemand_v2/{FP}/{f.relative_to(WORK)}")
print(len(new_api), "new embedding/QA cache entries exported")
target = Path("/kaggle/working" if KAGGLE else DRIVE) / f"{TAG}_export.zip"
shutil.copy(f"/tmp/{TAG}_export.zip", target)
print("export:", target, round(target.stat().st_size / 1e6, 1), "MB  (unzip at the repo root to bring results and caches home)")